In [ ]:
import torch
from torch import nn 
from torch import optim
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy

### Linear Regressor Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy

# 1. Dataset Generation
torch.manual_seed(204)

batch_size = 1000
feature_dim = 10 

X = torch.randn((batch_size, feature_dim))
weights = torch.randn(feature_dim)
bias = torch.randn(1)

# Added noise for a realistic regression task
noise = torch.randn(batch_size, 1) * 0.1 
Y = torch.matmul(X, weights.unsqueeze(1)) + bias + noise

# Split data (750 Train, 150 Val, 100 Test)
X_train, X_val, X_test = torch.split(X, [750, 150, 100])
Y_train, Y_val, Y_test = torch.split(Y, [750, 150, 100])

# 2. Model Definition
class LinearRegressor(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.linear_layer = nn.Linear(feature_dim, 1)

    def forward(self, X):
        return self.linear_layer(X)

regressor = LinearRegressor(feature_dim)

# 3. Training Setup
epochs = 1000
learning_rate = 0.01
optimizer = optim.Adam(regressor.parameters(), lr=learning_rate)
loss_func = nn.MSELoss()

best_val_loss = float("inf")
patience = 5
counter = 0 
best_state = None

# 4. Training Loop
for i in range(epochs):
    # --- Training Phase ---
    regressor.train()
    optimizer.zero_grad()
    
    train_predicted = regressor(X_train)
    train_loss = loss_func(train_predicted, Y_train)
    
    train_loss.backward()
    optimizer.step()

    # --- Validation Phase ---
    regressor.eval()
    with torch.no_grad():
        val_predicted = regressor(X_val)
        val_loss = loss_func(val_predicted, Y_val)
    
    val = val_loss.item()
    
    # --- Early Stopping Logic ---
    if val < best_val_loss:
        best_val_loss = val
        counter = 0 
        best_state = deepcopy(regressor.state_dict())
    else:
        counter += 1

    if i % 10 == 0:
        print(f"Epoch {i:3d} | Train Loss: {train_loss.item():.4f} | Val Loss: {val:.4f}")
        
    if counter >= patience:
        print(f"Early stopping at epoch {i}")
        break

# 5. Testing Phase
if best_state is not None:
    regressor.load_state_dict(best_state)
    print("Restored best model weights.")

regressor.eval()
with torch.no_grad():
    test_predicted = regressor(X_test)
    test_loss = loss_func(test_predicted, Y_test)

    print(f"Final Test Loss: {test_loss.item():.4f}")


## Logistic Regression Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy

# 1. Data Generation
torch.manual_seed(204)
batch_size = 1024 
input_dim = 8

X = torch.randn((batch_size, input_dim))
weights = torch.randn(input_dim)
bias = torch.randn(1)
noise = torch.randn((batch_size, 1))

# FIX: Add noise to the logits before thresholding to make it a realistic ML problem
logits = torch.matmul(X, weights.unsqueeze(1)) + bias + noise
Y = (torch.sigmoid(logits) >= 0.5).float().view(-1, 1)

X_train, X_val, X_test = torch.split(X, [750, 150, 124]) 
Y_train, Y_val, Y_test = torch.split(Y, [750, 150, 124])

# 2. Model
class LogisticRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear_layer = nn.Linear(input_dim, 1)

    def forward(self, X):
        # Returns Logits (unnormalized scores)
        return self.linear_layer(X)

logistic_regressor = LogisticRegressor(input_dim) 

# 3. Setup
optimizer = optim.Adam(logistic_regressor.parameters(), lr=0.01)
loss_func = nn.BCEWithLogitsLoss() # Handles Sigmoid internally

epochs = 1000 
patience = 20 
counter = 0 
min_loss = float("inf")
best_state = None 

# Helper function to calculate accuracy
def calculate_accuracy(y_pred_logits, y_true):
    # Apply sigmoid to convert logits to probabilities
    probs = torch.sigmoid(y_pred_logits)
    # Round to get 0 or 1
    predictions = probs.round()
    correct = (predictions == y_true).float()
    return correct.mean()

# 4. Training Loop
for i in range(epochs):
    # --- Train ---
    logistic_regressor.train()
    optimizer.zero_grad()

    predicted_train = logistic_regressor(X_train)
    train_loss = loss_func(predicted_train, Y_train)

    train_loss.backward()
    optimizer.step()

    # --- Validation ---
    logistic_regressor.eval()
    with torch.no_grad():
        predicted_val = logistic_regressor(X_val)
        val_loss = loss_func(predicted_val, Y_val)
        val_acc = calculate_accuracy(predicted_val, Y_val)

        val = val_loss.item()
        
        # Early Stopping Check
        if val < min_loss:
            min_loss = val 
            best_state = deepcopy(logistic_regressor.state_dict())
            counter = 0 
        else:
            counter += 1

        if i % 10 == 0:
            print(f"Epoch {i:3d} | Train Loss: {train_loss.item():.4f} | Val Loss: {val:.4f} | Val Acc: {val_acc:.4f}")
            
        if counter >= patience:
            print(f"Early stopping at epoch {i}")
            break

# 5. Restore Best Weights and Test
if best_state is not None:
    logistic_regressor.load_state_dict(best_state)
    print("Restored best model weights.")

logistic_regressor.eval()
with torch.no_grad():
    predicted_test = logistic_regressor(X_test)
    test_loss = loss_func(predicted_test, Y_test)
    test_acc = calculate_accuracy(predicted_test, Y_test)

    print(f"Final Test Loss: {test_loss.item():.4f}")
    print(f"Final Test Accuracy: {test_acc:.4f}")


## Implement MLP with multi class classification

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy

# 1. Setup & Data Generation
feature_dim = 10 
batch_size = 2048 
hidden_dim = 256
num_classes = 3 

torch.manual_seed(204)

# Generate synthetic data
X = torch.randn((batch_size, feature_dim))

# Simulate a non-linear relationship for the labels
weight1 = torch.randn((feature_dim, hidden_dim))
weight2 = torch.randn((hidden_dim, num_classes))
bias1 = torch.randn(1)
bias2 = torch.randn(1) 

# Create logits and then convert to class labels (0, 1, 2)
Y_logits = torch.relu(X @ weight1 + bias1) @ weight2 + bias2 
labels = torch.argmax(Y_logits, dim=1)

# Split Data
# Convention: Train (for optimization), Val (for early stopping), Test (final check)
# Renamed from your code to match standard ML conventions
X_train, X_val, X_test = torch.split(X, [1640, 205, 203])
Y_train, Y_val, Y_test = torch.split(labels, [1640, 205, 203])

# 2. Model Definition
class MultiClassifier(nn.Module):
    def __init__(self, feature_dim, hidden_dim, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim, bias=True),
            nn.ReLU(),
            # Output layer must have 'num_classes' neurons
            # No Softmax here because CrossEntropyLoss handles it
            nn.Linear(hidden_dim, num_classes, bias=True)     
        )
        
    def forward(self, X):
        return self.layers(X)

multiclassifier = MultiClassifier(feature_dim, hidden_dim, num_classes)

# 3. Training Setup
optimizer = optim.Adam(multiclassifier.parameters(), lr=0.0005)
loss_func = nn.CrossEntropyLoss() # Expects Logits + Class Indices
epochs = 2000 
patience = 30 
counter = 0 
best_state = None 
best_loss = float("inf")

# Helper for accuracy
def get_accuracy(logits, targets):
    predictions = torch.argmax(logits, dim=1)
    correct = (predictions == targets).sum().item()
    return correct / targets.size(0)

# 4. Training Loop
for i in range(epochs):
    # --- Train ---
    multiclassifier.train() # Fixed typo here
    optimizer.zero_grad()
    
    predicted_train = multiclassifier(X_train)
    loss_train = loss_func(predicted_train, Y_train)

    loss_train.backward()
    optimizer.step()

    # --- Validation (Early Stopping) ---
    multiclassifier.eval() # Fixed typo here
    with torch.no_grad():
        predicted_val = multiclassifier(X_val) # Using X_val here
        loss_val = loss_func(predicted_val, Y_val)
        val = loss_val.item()
        
        if val < best_loss:
            best_loss = val
            best_state = deepcopy(multiclassifier.state_dict())
            counter = 0
        else:
            counter += 1

    if i % 10 == 0:
        train_acc = get_accuracy(predicted_train, Y_train)
        val_acc = get_accuracy(predicted_val, Y_val)
        print(f"Epoch {i:4d} | Train Loss: {loss_train.item():.4f} (Acc: {train_acc:.2f}) | Val Loss: {val:.4f} (Acc: {val_acc:.2f})")

    if counter == patience:
        print(f"Early stopping at epoch {i}")
        break

# 5. Final Test
if best_state is not None:
    multiclassifier.load_state_dict(best_state)
    print("Restored best model weights.")

multiclassifier.eval() # Fixed typo here
with torch.no_grad():
    # Using X_test here for final evaluation
    prob_test = multiclassifier(X_test)
    loss_test = loss_func(prob_test, Y_test)
    accuracy = get_accuracy(prob_test, Y_test)
    
    print(f"Final Test Loss: {loss_test.item():.4f}; Accuracy: {accuracy:.4f}")


## Tokenization for NLP
1. WordPiece (BERT-style)
WordPiece builds a vocabulary by greedily maximizing the likelihood of the training corpus under a language model. Subwords that aren't the start of a word get a ## prefix.
Training algorithm:

Start with a character-level vocabulary (all individual characters + special tokens).
Iteratively merge the pair (A, B) that maximizes:

score(A,B)=count(AB)count(A)×count(B)

This differs from BPE — instead of raw frequency, it favors pairs where the merged token is more informative than its parts alone.

Repeat until vocabulary size is reached.

Inference (tokenization):
WordPiece uses a longest-match-first (greedy left-to-right) strategy per word:

Try the longest prefix that exists in the vocab.
Mark remaining subwords with ##.
If no valid split exists → [UNK].

In [44]:
from collections import defaultdict

class WordPieceTrainer:
    def __init__(self, vocab_size=1000, special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]):
        self.vocab_size = vocab_size
        self.special_tokens = special_tokens
        self.vocab = {}

    def get_word_freqs(self, corpus: list[str]) -> dict:
        """Count word frequencies in corpus."""
        word_freqs = defaultdict(int)
        for text in corpus:
            for word in text.lower().split():
                word_freqs[word] += 1
        return word_freqs

    def word_to_chars(self, word_freqs: dict) -> dict:
        """
        Split each word into characters.
        Non-first characters get '##' prefix.
        e.g. 'hello' -> ['h', '##e', '##l', '##l', '##o']
        """
        splits = {}
        for word in word_freqs:
            splits[word] = [c if i == 0 else f"##{c}" for i, c in enumerate(word)]
        return splits

    def build_initial_vocab(self, word_freqs: dict) -> set:
        """Initialize vocab with all unique characters."""
        vocab = set(self.special_tokens)
        for word in word_freqs:
            for i, char in enumerate(word):
                token = char if i == 0 else f"##{char}"
                vocab.add(token)
        return vocab

    def compute_pair_scores(self, splits: dict, word_freqs: dict) -> dict:
        """
        Score each adjacent pair using WordPiece criterion:
            score(A, B) = freq(AB) / (freq(A) * freq(B))
        """
        token_freqs = defaultdict(int)
        pair_freqs = defaultdict(int)

        for word, freq in word_freqs.items():
            split = splits[word]
            if len(split) == 1:
                token_freqs[split[0]] += freq
                continue
            for i, token in enumerate(split):
                token_freqs[token] += freq
                if i < len(split) - 1:
                    pair_freqs[(split[i], split[i+1])] += freq

        scores = {}
        for pair, freq in pair_freqs.items():
            denom = token_freqs[pair[0]] * token_freqs[pair[1]]
            scores[pair] = freq / denom if denom > 0 else 0
        return scores

    def merge_pair(self, pair: tuple, splits: dict) -> dict:
        """Merge a token pair across all words."""
        a, b = pair
        merged = a + b.replace("##", "")  # '##el' -> 'el', merge 'h' + 'el' = 'hel'
        for word in splits:
            split = splits[word]
            new_split = []
            i = 0
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(merged)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            splits[word] = new_split
        return splits

    def train(self, corpus: list[str]) -> dict:
        """Full WordPiece vocab training."""
        word_freqs = self.get_word_freqs(corpus)
        splits = self.word_to_chars(word_freqs)
        vocab = self.build_initial_vocab(word_freqs)

        print(f"Initial vocab size: {len(vocab)}")

        while len(vocab) < self.vocab_size:
            scores = self.compute_pair_scores(splits, word_freqs)
            if not scores:
                break
            best_pair = max(scores, key=scores.get)
            splits = self.merge_pair(best_pair, splits)
            new_token = best_pair[0] + best_pair[1].replace("##", "")
            vocab.add(new_token)

            if len(vocab) % 100 == 0:
                print(f"Vocab size: {len(vocab)} | Last merge: {best_pair} -> '{new_token}'")

        # Assign integer IDs
        self.vocab = {token: idx for idx, token in enumerate(sorted(vocab))}
        print(f"\nFinal vocab size: {len(self.vocab)}")
        return self.vocab

# corpus = [
#     "the quick brown fox jumps over the lazy dog",
#     "hello world this is a tokenizer",
#     "wordpiece tokenization is used in bert",
#     "subword tokenization helps with rare words",
# ]

# trainer = WordPieceTrainer(vocab_size=200)
# vocab = trainer.train(corpus)
# print("\nSample vocab entries:", list(vocab.items())[:20])

In [51]:
def wordpiece_tokenizer(word, vocab):
    tokens = []
    start = 0 
    while start < len(word):
        end = len(word) 
        found = None 
        while start < end:
            substr = word[start:end] 
            candidate = substr if start == 0 else "##" + substr 
            if candidate in vocab:
                found = candidate
                break
            end -= 1
        if found is None:
            return ["[UNK]"]
        tokens.append(found)
        start = end
    return tokens 


# Example using merges from BPETrainer
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "hello world tokenization with bpe merges",
]

trainer = WordPieceTrainer(vocab_size=200)
vocab = trainer.train(corpus)

print(wordpiece_tokenizer("hello", set(vocab.keys())))
print(wordpiece_tokenizer("quick", set(vocab.keys())))

Initial vocab size: 38

Final vocab size: 91
['hello']
['quick']


## BPE - Byte Pair Encoding (GPT-style) 

BPE builds vocabulary by raw frequency of adjacent pairs. No probabilistic scoring — just count and merge.

Training algorithm:

Pre-tokenize corpus into words, represent each word as a sequence of characters (+ end-of-word marker, e.g. </w>).
Count frequency of all adjacent symbol pairs across all words.
Merge the most frequent pair → add merged token to vocab.
Re-count. Repeat until vocab size is reached.

In [46]:
from collections import defaultdict
import re

class BPETrainer:
    def __init__(self, vocab_size=1000, special_tokens=["<|endoftext|>", "<|pad|>"]):
        self.vocab_size = vocab_size
        self.special_tokens = special_tokens
        self.merges = []   # ordered list of merges — needed for inference
        self.vocab = {}

    def get_word_freqs(self, corpus: list[str]) -> dict:
        """
        Count word frequencies, representing each word as
        space-separated characters + end-of-word marker </w>.
        e.g. "hello" -> "h e l l o </w>"
        """
        word_freqs = defaultdict(int)
        for text in corpus:
            for word in text.lower().split():
                word_freqs[" ".join(list(word)) + " </w>"] += 1
        return word_freqs

    def build_initial_vocab(self, word_freqs: dict) -> set:
        """Initialize vocab with all unique characters + special tokens."""
        vocab = set(self.special_tokens)
        for word in word_freqs:
            for symbol in word.split():
                vocab.add(symbol)
        return vocab

    def get_pairs(self, word_freqs: dict) -> dict:
        """Count frequency of every adjacent symbol pair across all words."""
        pairs = defaultdict(int)
        for word, freq in word_freqs.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs

    def merge_pair(self, pair: tuple, word_freqs: dict) -> dict:
        """Merge the best pair in every word that contains it."""
        new_word_freqs = {}
        bigram = re.escape(" ".join(pair))
        pattern = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
        replacement = "".join(pair)
        for word, freq in word_freqs.items():
            new_word = pattern.sub(replacement, word)
            new_word_freqs[new_word] = freq
        return new_word_freqs

    def train(self, corpus: list[str]) -> tuple[dict, list]:
        """Full BPE vocab training. Returns (vocab, merges)."""
        word_freqs = self.get_word_freqs(corpus)
        vocab = self.build_initial_vocab(word_freqs)

        print(f"Initial vocab size: {len(vocab)}")

        while len(vocab) < self.vocab_size:
            pairs = self.get_pairs(word_freqs)
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            word_freqs = self.merge_pair(best_pair, word_freqs)

            new_token = "".join(best_pair)
            vocab.add(new_token)
            self.merges.append(best_pair)

            if len(vocab) % 100 == 0:
                print(f"Vocab size: {len(vocab)} | Last merge: {best_pair} -> '{new_token}'")

        # Assign integer IDs
        self.vocab = {token: idx for idx, token in enumerate(sorted(vocab))}
        print(f"\nFinal vocab size: {len(self.vocab)}")
        print(f"Total merges learned: {len(self.merges)}")
        return self.vocab, self.merges



# corpus = [
#     "the quick brown fox jumps over the lazy dog",
#     "hello world this is a tokenizer",
#     "byte pair encoding is used in gpt models",
#     "subword tokenization helps with rare words",
# ]

# trainer = BPETrainer(vocab_size=200)
# vocab, merges = trainer.train(corpus)
# print("\nSample vocab entries:", list(vocab.items())[:20])
# print("\nFirst 10 merges:", merges[:10])

In [50]:
def bpe_tokenizer(word: str, merges: list[tuple], vocab: set) -> list[str]:
    """
    Correct BPE inference: start from characters, apply merges in order.
    merges: ordered list of (tokenA, tokenB) pairs from training.
    """
    # Step 1: split word into characters + end-of-word marker
    tokens = list(word) + ["</w>"]

    if len(tokens) == 1:
        return tokens

    # Step 2: apply each learned merge in training order
    for merge_a, merge_b in merges:
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] == merge_a and tokens[i+1] == merge_b:
                tokens = tokens[:i] + [merge_a + merge_b] + tokens[i+2:]
                # don't increment i — new token might be part of next merge
            else:
                i += 1

    # Step 3: fallback — any token not in vocab becomes [UNK]
    tokens = [t if t in vocab else "[UNK]" for t in tokens]
    return tokens


# # Example using merges from BPETrainer
# corpus = [
#     "the quick brown fox jumps over the lazy dog",
#     "hello world tokenization with bpe merges",
# ]

# trainer = BPETrainer(vocab_size=200)
# vocab, merges = trainer.train(corpus)

# print(bpe_tokenizer("hello", merges, set(vocab.keys())))
# # -> ['hello</w>'] if fully merged, or ['hel', 'lo</w>'] etc depending on corpus
# print(bpe_tokenizer("quick", merges, set(vocab.keys())))

## Byte-Level BPE (GPT-2 / preventing OOV)
The core insight: represent text as raw bytes (0–255) before any tokenization. This gives a base vocabulary of 256 symbols — every possible byte — so OOV is structurally impossible.
GPT-2 uses this. Llama, Falcon, Mistral all followed.
How it works:

Step 1: encode text to bytes, map each byte to a unicode character, GPT-2 uses a specific 256-char lookup table so bytes are printable. 

Step2: Then run standard BPE on top of this byte representation.

**Why this handles multilingual + multimodal:**
- Chinese, Arabic, emoji, binary file headers — everything is just bytes.
- No language-specific pre-tokenization needed.
- Worst case: a rare character = 3–4 byte tokens. Common chars still get merged efficiently.


## Putting it Together — Full Pipeline Sketch
```
Raw text
   │
   ▼
[Pre-tokenization] — split on whitespace/punctuation
   │
   ▼
[Byte encoding] — text → bytes → unicode chars (for byte-level)
   │
   ▼
[BPE or WordPiece training] — build merge table / vocab
   │
   ▼
[Tokenizer inference] — apply merges or longest-match
   │
   ▼
[Special tokens] — [CLS], [SEP], <|endoftext|>, etc.
   │
   ▼
Integer IDs → model input

In [88]:
from collections import defaultdict
import re

class ByteLevelBPETrainer:
    def __init__(self, vocab_size=1000, special_tokens=["<|endoftext|>", "<|pad|>", "<|unk|>"]):
        self.vocab_size = vocab_size
        self.special_tokens = special_tokens
        self.merges = []
        self.vocab = {}
        self.byte_encoder = self._build_byte_encoder()
        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}

    def _build_byte_encoder(self) -> dict[int, str]:
        """
        Map every byte (0-255) to a unique printable unicode character.
        This is exactly GPT-2's mapping — ensures all 256 bytes are
        visible/printable so BPE can operate on them cleanly.
        """
        # Bytes that already map to nice printable chars
        bs = (
            list(range(ord('!'), ord('~') + 1)) +   # 33–126
            list(range(ord('¡'), ord('¬') + 1)) +   # 161–172
            list(range(ord('®'), ord('ÿ') + 1))      # 174–255
        )
        cs = bs[:]
        # Remaining bytes get mapped to chars starting at 256
        n = 0
        for b in range(256):
            if b not in bs:
                bs.append(b)
                cs.append(256 + n)
                n += 1
        return dict(zip(bs, [chr(c) for c in cs]))

    def text_to_byte_chars(self, text: str) -> list[str]:
        """Convert raw text to a list of unicode chars representing bytes."""
        return [self.byte_encoder[b] for b in text.encode("utf-8")]

    def get_word_freqs(self, corpus: list[str]) -> dict:
        """
        Byte-encode each word, then represent as space-separated
        byte-chars for BPE processing.
        """
        word_freqs = defaultdict(int)
        for text in corpus:
            for word in text.split():
                # Encode word to bytes, prefix first char with Ġ (space marker)
                byte_chars = self.text_to_byte_chars(word)
                # GPT-2 marks word boundaries with Ġ (byte 32 = space, shifted)
                key = " ".join(byte_chars)
                word_freqs[key] += 1
        return word_freqs

    def build_initial_vocab(self, word_freqs: dict) -> set:
        """
        Initial vocab = all 256 byte characters + special tokens.
        This GUARANTEES zero OOV — every possible input maps to known tokens.
        """
        vocab = set(self.special_tokens)
        # Add all 256 possible byte representations
        for byte_char in self.byte_encoder.values():
            vocab.add(byte_char)
        print(f"  Base byte vocab: {len(self.byte_encoder)} byte symbols")
        return vocab

    def get_pairs(self, word_freqs: dict) -> dict:
        pairs = defaultdict(int)
        for word, freq in word_freqs.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs

    def merge_pair(self, pair: tuple, word_freqs: dict) -> dict:
        new_word_freqs = {}
        bigram = re.escape(" ".join(pair))
        pattern = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
        replacement = "".join(pair)
        for word, freq in word_freqs.items():
            new_word = pattern.sub(replacement, word)
            new_word_freqs[new_word] = freq
        return new_word_freqs

    def train(self, corpus: list[str]) -> tuple[dict, list]:
        """Full byte-level BPE training. Returns (vocab, merges)."""
        word_freqs = self.get_word_freqs(corpus)
        vocab = self.build_initial_vocab(word_freqs)

        print(f"Initial vocab size: {len(vocab)}")

        while len(vocab) < self.vocab_size:
            pairs = self.get_pairs(word_freqs)
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            word_freqs = self.merge_pair(best_pair, word_freqs)

            new_token = "".join(best_pair)
            vocab.add(new_token)
            self.merges.append(best_pair)

            if len(vocab) % 100 == 0:
                print(f"Vocab size: {len(vocab)} | Last merge: {best_pair} -> '{new_token}'")

        self.vocab = {token: idx for idx, token in enumerate(sorted(vocab))}
        print(f"\nFinal vocab size: {len(self.vocab)}")
        print(f"Total merges learned: {len(self.merges)}")
        return self.vocab, self.merges

    def decode_token(self, token: str) -> str:
        """Convert a token (byte chars) back to readable text."""
        return bytes([self.byte_decoder[c] for c in token]).decode("utf-8", errors="replace")


# ── Example usage ──────────────────────────────────────────────────────────────
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "hello world this is a tokenizer",
    "byte level BPE handles any language: 日本語 العربية 한국어",
    "even emoji work fine: 🚀 🎉 🤖",
    "and binary-ish content won't cause OOV errors",
]

trainer = ByteLevelBPETrainer(vocab_size=400)
vocab, merges = trainer.train(corpus)
# print("\nSample vocab entries:", list(vocab.items())[:20])
# print("\nFirst 10 merges:", merges[:10])

# # Demonstrate zero OOV: decode some tokens back
# print("\nDecoding sample tokens:")
# for token in list(vocab.keys())[260:270]:
#     try:
#         print(f"  '{token}' -> '{trainer.decode_token(token)}'")
#     except:
#         print(f"  '{token}' -> (special token)")

  Base byte vocab: 256 byte symbols
Initial vocab size: 259
Vocab size: 300 | Last merge: ('worl', 'd') -> 'world'

Final vocab size: 392
Total merges learned: 133


In [89]:
def byte_level_bpe_tokenizer(text: str, merges: list[tuple], vocab: set,
                              byte_encoder: dict) -> list[str]:
    """
    Byte-level BPE inference:
      1. Encode text to bytes → unicode chars
      2. Split on whitespace (each word tokenized independently)
      3. Apply BPE merges in order
      4. Return final token list
    """
    def tokenize_word(byte_chars: list[str]) -> list[str]:
        tokens = byte_chars[:]
        for merge_a, merge_b in merges:
            i = 0
            while i < len(tokens) - 1:
                if tokens[i] == merge_a and tokens[i+1] == merge_b:
                    tokens = tokens[:i] + [merge_a + merge_b] + tokens[i+2:]
                else:
                    i += 1
        return tokens

    all_tokens = []
    words = text.split(" ")

    for idx, word in enumerate(words):
        if not word:
            continue
        # Encode word to byte-chars
        byte_chars = [byte_encoder[b] for b in word.encode("utf-8")]
        word_tokens = tokenize_word(byte_chars)
        # Validate against vocab
        word_tokens = [t if t in vocab else "<|unk|>" for t in word_tokens]
        all_tokens.extend(word_tokens)

    return all_tokens


def decode_tokens(tokens: list[str], byte_decoder: dict,
                  special_tokens: set) -> str:
    """Convert byte-level BPE tokens back to a readable string."""
    byte_list = []
    for token in tokens:
        if token in special_tokens:
            continue
        byte_list.extend([byte_decoder[c] for c in token])
    return bytes(byte_list).decode("utf-8", errors="replace")


# ── Full round-trip example ────────────────────────────────────────────────────
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "hello world tokenization with byte level bpe",
    "multilingual: 日本語 العربية",
    "emoji too: 🚀 🎉",
]

trainer = ByteLevelBPETrainer(vocab_size=400)
vocab, merges = trainer.train(corpus)

byte_encoder = trainer.byte_encoder
byte_decoder = trainer.byte_decoder
special_tokens = set(trainer.special_tokens)

# Tokenize
test = "hello world 🚀"
tokens = byte_level_bpe_tokenizer(test, merges, set(vocab.keys()), byte_encoder)
print("Tokens:  ", tokens)

# Decode back
decoded = decode_tokens(tokens, byte_decoder, special_tokens)
print("Decoded: ", decoded)
# -> "hello world 🚀"

  Base byte vocab: 256 byte symbols
Initial vocab size: 259
Vocab size: 300 | Last merge: ('tokeniz', 'a') -> 'tokeniza'

Final vocab size: 355
Total merges learned: 96
Tokens:   ['hello', 'world', 'ðŁļĢ']
Decoded:  helloworld🚀


## embedding model with linear classifer  (NLP-style)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy

# --- 1. Robust Device Selection ---
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    # Check for Apple Silicon (M1/M2/M3)
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Running on: {device}")

# --- 2. Model Definitions (Unchanged) ---
class MaxPoolNLP(nn.Module):
    def __init__(self, vocab_size, embed_size, num_classes):
        super().__init__()
        self.embed_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_size, padding_idx=0)
        self.linear = nn.Linear(embed_size, num_classes, bias=True)

    def forward(self, input_ids, mask):
        embedding = self.embed_layer(input_ids)
        mask = mask.unsqueeze(-1)
        masked_embedding = embedding.masked_fill(mask==0, -float("inf"))
        max_pooled, _ = masked_embedding.max(dim=1) 

        # 3. Safety Handling (Optional but recommended):
        # If a sequence was entirely padding, max_pooled is now -inf.
        # Passing -inf to Linear layers causes NaNs. We clamp the RESULT, not the mask.
        # We replace -inf with 0.0 or a large negative number just to keep the math stable.
        max_pooled = torch.nan_to_num(max_pooled, nan=0.0, neginf=0.0)

        return self.linear(max_pooled) 

class MeanPoolNLP(nn.Module):
    def __init__(self, vocab_size, embed_size, num_classes):
        super().__init__()
        self.embed_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_size, padding_idx=0)
        self.linear = nn.Linear(embed_size, num_classes, bias=True)

    def forward(self, input_ids, mask):
        embedding = self.embed_layer(input_ids)
        mask = mask.unsqueeze(-1).float()
        masked_embedding = embedding * mask
        total_count = mask.sum(dim=1).clamp(min=1e-8)
        total_sum = masked_embedding.sum(dim=1) 
        mean_pooled = total_sum / total_count 
        return self.linear(mean_pooled)

def calculate_accuracy(predicted, label):
    return (predicted.argmax(dim=1) == label).float().mean().item()

# --- 3. Updated Training Function with Device Support ---
def train_test(model, loss_func, optimizer, train_loader, val_loader, epoch):
    # Move the entire model to the device (GPU/MPS/CPU)
    model = model.to(device)
    
    lowest_loss = float("inf")
    best_state = None
    patience = 20 
    count = 0
    
    for i in range(epoch):
        model.train()

        train_loss_accum = 0 
        train_acc_accum = 0
        train_batches = 0 
        
        # Iterate over batches
        for x_batch, mask_batch, y_batch in train_loader:
            # Move batch data to device
            x_batch = x_batch.to(device)
            mask_batch = mask_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            train_predicted = model(x_batch, mask_batch)
            train_loss = loss_func(train_predicted, y_batch)
            train_loss.backward()
            optimizer.step()

            train_loss_accum += train_loss.item()
            train_acc_accum += calculate_accuracy(train_predicted, y_batch)
            train_batches += 1

        avg_train_loss = train_loss_accum /train_batches
        avg_train_acc = train_acc_accum/train_batches 

        # Validation
        model.eval()
        val_loss_accum = 0
        val_acc_accum = 0 
        val_batches = 0
        
        with torch.no_grad():
            for x_val, mask_val, y_val in val_loader:
                x_val = x_val.to(device)
                mask_val = mask_val.to(device)
                y_val = y_val.to(device)
                
                val_predicted = model(x_val, mask_val)
                val_loss = loss_func(val_predicted, y_val)
                val_loss_accum += val_loss.item()
                val_acc_accum += calculate_accuracy(val_predicted, y_val)
                val_batches += 1
            
            avg_val_loss = val_loss_accum / val_batches
            avg_val_accuracy = val_acc_accum/val_batches

            if avg_val_loss < lowest_loss:
                lowest_loss = avg_val_loss 
                best_state = deepcopy(model.state_dict())
                count = 0
            else:
                count += 1

        if i % 10 == 0:
            print(f"Epoch {i}: train_loss: {avg_train_loss:.4f} (acc: {avg_train_acc:.2f}); val_loss: {avg_val_loss:.4f} (acc: {avg_val_accuracy:.2f})")

        if count == patience:
            print(f"Early stopping with epoch {i}")
            break
            
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def evaluate_model(model, loss_func, data_loader):
    model.eval() # Fixed syntax error from original code
    model = model.to(device)
    
    total_acc = 0
    total_loss = 0 
    batches = 0
    
    with torch.no_grad():
        for x, mask, y in data_loader:
            x, mask, y = x.to(device), mask.to(device), y.to(device)
            predicted = model(x, mask)
            total_loss += loss_func(predicted, y).item()
            total_acc += calculate_accuracy(predicted, y)
            batches += 1
        avg_loss = total_loss / batches 
    print(f"Final Accuracy: {total_acc/batches:.4f}; Final Loss: {avg_loss:.4f}")

# --- 4. Data Setup ---
torch.manual_seed(204)
batch_size = 1024 # You can lower this if you hit memory errors on GPU
seq_len = 10 
embed_size = 512 
vocab_size = 10000
num_classes = 3 

X = torch.randint(1, vocab_size, (batch_size, seq_len))
mask = torch.ones_like(X)

for i in range(batch_size):
    pad_len = torch.randint(0, seq_len//2, (1,)).item()
    if pad_len > 0:
        X[i,seq_len - pad_len:] = 0 
        mask[i, seq_len - pad_len:] = 0 

Y = torch.randint(0, num_classes, (batch_size,))

X_train, X_val, X_test = torch.split(X, [800,124,100])
Y_train, Y_val, Y_test = torch.split(Y, [800,124,100]) 
mask_train, mask_val, mask_test = torch.split(mask, [800,124,100])

# Create DataLoaders
train_data = TensorDataset(X_train, mask_train, Y_train)
val_data = TensorDataset(X_val, mask_val, Y_val)
test_data = TensorDataset(X_test, mask_test, Y_test)

# Note: shuffle=True is important for training!
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

max_pool_nlp = MaxPoolNLP(vocab_size, embed_size, num_classes)
mean_pool_nlp = MeanPoolNLP(vocab_size, embed_size, num_classes)

epochs = 1000
optimizer_max = optim.Adam(max_pool_nlp.parameters(), lr= 0.001)
optimizer_mean = optim.Adam(mean_pool_nlp.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()

print("Training Max Pool Model...")
best_max_nlp = train_test(max_pool_nlp, loss_func, optimizer_max, train_loader, val_loader, epochs)

evaluate_model(best_max_nlp, loss_func, test_loader)

print("Training Mean Pool Model...")
best_mean_nlp = train_test(mean_pool_nlp, loss_func, optimizer_mean, train_loader, val_loader, epochs)

evaluate_model(best_mean_nlp, loss_func, test_loader)


# 4 Layer MLP Classifier in PyTorch (NLP Embedding)
Kaiming/Xavier initialization
Understanding depth challenges

* Add batch normalization or layer normalization
* dropout and regularization (L1/L2)
Gradient monitoring, learning rate scheduling, loss curves

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from copy import deepcopy

# --- 1. Robust Device Selection ---
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    # Check for Apple Silicon (M1/M2/M3)
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Running on: {device}")

## Define the MLPClassifer 
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, hidden_dim3, num_classes):
        # Renamed embed_layer to input_dim for clarity
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.BatchNorm1d(hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.BatchNorm1d(hidden_dim2),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(hidden_dim2, hidden_dim3),
            nn.BatchNorm1d(hidden_dim3),
            nn.ReLU(),
            nn.Linear(hidden_dim3, num_classes)
        )
        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

    def forward(self, embeddings):
        return self.layers(embeddings)
                
# define datasets      
torch.manual_seed(204)
data_size = 10000
embed_dim = 384 
hidden_dim1 = embed_dim * 4
hidden_dim2 = hidden_dim1 * 2
hidden_dim3 = hidden_dim2 
num_classes = 3 

X = torch.randn((data_size, embed_dim))
Y = torch.randint(0, num_classes, (data_size,))
X_train, X_val, X_test = torch.split(X, [8000, 1000, 1000])
Y_train, Y_val, Y_test = torch.split(Y, [8000, 1000, 1000])

train_data = TensorDataset(X_train, Y_train)
val_data = TensorDataset(X_val, Y_val)
test_data_set = TensorDataset(X_test, Y_test) # Renamed to avoid overwriting

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64)
test_loader = DataLoader(test_data_set, batch_size=64) # FIX: Assigned to test_loader

# train and validate log in tensorboard
writer = SummaryWriter(log_dir="runs/mlp_stability")
model = MLPClassifier(embed_dim, hidden_dim1, hidden_dim2, hidden_dim3, num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) 
l1_lambda = 1e-5
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5) 
patience = 10 
epochs = 20 
counter = 0 
best_state = None 
lowest_loss = float("inf") 
global_step = 0 

# Note: These magic commands only work in Jupyter Notebooks/Colab
# %load_ext tensorboard
# %tensorboard --logdir runs

for i in range(epochs):
    model.train()
    train_loss_acc = 0 

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        predicted = model(x)
        
        # L1 Regularization
        l1_norm = sum(p.abs().sum() for p in model.parameters())
        loss = criterion(predicted, y) + l1_lambda * l1_norm 
        loss.backward()

        # Gradient Norm Monitoring
        total_grad_norm = 0.0 
        for p in model.parameters():
            if p.grad is not None: # FIX: Added safety check
                param_norm = p.grad.data.norm(2) 
                total_grad_norm += param_norm.item() ** 2 
        total_grad_norm = total_grad_norm ** 0.5 

        optimizer.step() 

        writer.add_scalar("train/loss", loss.item(), global_step)
        writer.add_scalar("train/grad_norm", total_grad_norm, global_step)

        train_loss_acc += loss.item()
        global_step += 1

    avg_train_loss = train_loss_acc / len(train_loader) 

    model.eval()
    with torch.no_grad():
        val_loss_acc = 0 

        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            predicted = model(x)
            loss = criterion(predicted, y)
            val_loss_acc += loss.item()

        avg_val_loss = val_loss_acc / len(val_loader)

    writer.add_scalar("val/avg_loss", avg_val_loss, i) 
    writer.add_scalar("train/avg_loss", avg_train_loss, i) 

    # ---- Learning rate ----
    current_lr = optimizer.param_groups[0]["lr"]
    writer.add_scalar("lr", current_lr, i)

    print(
        f"Epoch {i+1}/{epochs} | " # FIX: Changed num_epochs to epochs
        f"avg Train Loss: {avg_train_loss:.4f} | "
        f"avg val Loss: {avg_val_loss:.4f} | "
        f"LR: {current_lr:.5f}"
    )

    # FIX: Changed &lt; to <
    if avg_val_loss < lowest_loss:
        lowest_loss = avg_val_loss
        best_state = deepcopy(model.state_dict())
        counter = 0 
    else:
        counter += 1

    if counter == patience:
        print(f"early stopping at epoch {i+1}")
        break 

    # learning rate update
    scheduler.step()

writer.close()

if best_state is not None:
    model.load_state_dict(best_state) 
    
# evaluate 
model.eval()
with torch.no_grad():
    test_loss_acc = 0 
    for x, y in test_loader: # FIX: Now test_loader is defined
        x = x.to(device)
        y = y.to(device)
        predicted = model(x)
        test_loss_acc += criterion(predicted, y) 
    avg_test_loss = test_loss_acc / len(test_loader)

print(f"test avg loss is {avg_test_loss:.4f}") 
%load_ext tensorboard
%tensorboard --logdir runs

## CNN (1D conv over tokens)

convolution layers for text 
pooling strategies (max, avg)
multi filter sizes
Gradient monitoring, learning rate scheduling, loss curves
overfitting prevention, training stability 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from copy import deepcopy

# --- 1. Define Class ---
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx, num_filters, kernels, dropout, num_classes):
        super().__init__()
        # Embedding: (Batch, Seq_Len) -> (Batch, Seq_Len, Embed_Dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        
        # ModuleList allows us to register a list of layers properly
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k)
            for k in kernels
        ])
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        
        # The input to FC is num_filters * number of different kernel sizes
        self.fc = nn.Linear(num_filters * len(kernels), num_classes)

    def forward(self, input_ids):
        # 1. Embed
        embed = self.embedding(input_ids) # (B, T, D)
        
        # 2. Transpose for Conv1d: (B, T, D) -> (B, D, T)
        # Conv1d expects (Batch, Channels, Length)
        embed = embed.transpose(1, 2) 
        
        conv_outputs = []
        for conv in self.convs:
            # Apply Convolution: (B, D, T) -> (B, F, T_out)
            conv_output = conv(embed)
            
            # Apply Activation
            conv_output = self.relu(conv_output)
            
            # Global Max Pooling over time dimension: (B, F, T_out) -> (B, F)
            conv_output = torch.max(conv_output, dim=2).values 
            conv_outputs.append(conv_output)
        
        # 3. Concatenate all filter outputs: (B, F * len(kernels))
        outputs = torch.cat(conv_outputs, dim=1) 
        
        # 4. Dropout and FC
        outputs = self.dropout(outputs)
        return self.fc(outputs) 

# --- 2. Device Setup ---
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"Running on: {device}")

# --- 3. Data Generation ---
data_size = 10000
batch_size = 64
seq_len = 20
vocab_size = 10000
padding_idx = 0 

# Generate random data
X = torch.randint(1, vocab_size, (data_size, seq_len))

# Add random padding (simulating variable length sequences)
padding_lens = torch.randint(0, seq_len//2, (data_size,))
for i, pad_len in enumerate(padding_lens):
    pad = pad_len.item()
    if pad > 0:
        X[i, (seq_len - pad):] = padding_idx 

# Create labels: Class 1 if any token ID >= 500, else Class 0
Y = (X >= 500).any(dim=1).long()

# Shuffle and Split
# Standard Split: 80% Train, 10% Validation, 10% Test
perm = torch.randperm(data_size)
X = X[perm]
Y = Y[perm]

X_train, X_val, X_test = torch.split(X, [8000, 1000, 1000])
Y_train, Y_val, Y_test = torch.split(Y, [8000, 1000, 1000])

train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, Y_val), batch_size=batch_size) # Used for early stopping
test_loader = DataLoader(TensorDataset(X_test, Y_test), batch_size=batch_size) # Used for final check

# --- 4. Model Initialization ---
embed_dim = 128
kernel_sizes = (3, 4, 5) # Using multiple kernel sizes is common in TextCNN
dropout = 0.5
num_filters = 50

model = TextCNN(
    vocab_size, 
    embed_dim, 
    padding_idx=padding_idx, 
    num_filters=num_filters, 
    kernels=kernel_sizes, 
    dropout=dropout, 
    num_classes=2
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5) 
writer = SummaryWriter(log_dir="runs/textcnn_experiment")

# --- 5. Helper Functions ---
def get_total_accuracy(logits, labels):
    predicted = torch.argmax(logits, dim=1)
    return torch.sum((predicted == labels).float()).item()

def compute_grad_norm(model):
    total_norm = 0 
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item()**2 
    return total_norm**0.5

# --- 6. Training Loop ---
epochs = 20
patience = 5
best_state = None
lowest_loss = float("inf")
counter = 0 
global_step = 0

for epoch in range(epochs):
    # -- TRAIN --
    model.train()
    total_train_acc = 0
    
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        
        optimizer.zero_grad()
        pred = model(x) 
        loss = criterion(pred, y)
        loss.backward()
        
        grad_norm = compute_grad_norm(model)
        optimizer.step()
        
        total_train_acc += get_total_accuracy(pred, y) 
        
        writer.add_scalar("train loss", loss.item(), global_step)
        writer.add_scalar("grad norm", grad_norm, global_step)
        global_step += 1
        
    avg_train_accuracy = total_train_acc / len(X_train)

    # -- VALIDATION (For Early Stopping) --
    model.eval()
    total_val_loss = 0
    total_val_acc = 0 
    
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            pred = model(x) 
            loss = criterion(pred, y) 
            
            # FIX: Use loss.item() to avoid memory leak
            total_val_loss += loss.item()
            total_val_acc += get_total_accuracy(pred, y) 

    avg_val_loss = total_val_loss / len(val_loader)
    avg_val_accuracy = total_val_acc / len(X_val)

    # Logging
    writer.add_scalar("avg_train_acc", avg_train_accuracy, epoch)
    writer.add_scalar("avg_val_acc", avg_val_accuracy, epoch) 
    writer.add_scalar("avg_val_loss", avg_val_loss, epoch) 
    writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)

    print(f"Epoch {epoch+1} | Train Acc: {avg_train_accuracy:.4f} | Val Acc: {avg_val_accuracy:.4f} | Val Loss: {avg_val_loss:.4f}")

    # -- Early Stopping Logic --
    # FIX: Replaced &lt; with <
    if avg_val_loss < lowest_loss:
        lowest_loss = avg_val_loss 
        best_state = deepcopy(model.state_dict())
        counter = 0
        print("  -> New best model found.")
    else:
        counter += 1
        print(f"  -> No improvement. Patience {counter}/{patience}")

    if counter == patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

    scheduler.step(avg_val_loss)

writer.close()

# --- 7. Final Evaluation on Test Set ---
if best_state is not None:
    model.load_state_dict(best_state)    

model.eval()
test_loss = 0
test_acc = 0 

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)
        pred = model(x)
        loss = criterion(pred, y)
        
        test_loss += loss.item() 
        test_acc += get_total_accuracy(pred, y) 

print("-" * 30)
print(f"Final Test Loss: {test_loss/len(test_loader):.4f}")
print(f"Final Test Acc:  {test_acc/len(X_test):.4f}")
print("-" * 30)

%load_ext tensorboard
%tensorboard --logdir runs --port 6007

# RNN/GRU/LSTM


vanilla RNN 
GRU
LSTM 
Bidirectional variants
Gradient monitoring, learning rate scheduling, loss curves
overfitting prevention, training stability 

In [ ]:
from copy import deepcopy
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_size, padding_idx, memory_cell_class, hidden_dim, num_layers, bidirectional, representation, num_classes):
        super().__init__()
        self.padding_idx = padding_idx
        self.bidirectional = bidirectional
        self.representation = representation
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=padding_idx)
        
        # Instantiate the passed class (nn.GRU or nn.LSTM)
        self.memory_cell = memory_cell_class(
            embed_size,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=bidirectional,
            batch_first=True
        )
        
        multiplier = 2 if bidirectional else 1
        self.FC = nn.Linear(multiplier * hidden_dim, num_classes)

    def forward(self, input_ids, lengths):
        # input_ids: (B, T)
        embedding = self.embedding(input_ids) 
        
        # Pack
        packed_embedding = pack_padded_sequence(
            embedding,
            lengths=lengths.cpu(), # Must be on CPU for packing
            batch_first=True,
            enforce_sorted=False
        )
        
        # Forward pass through RNN
        if isinstance(self.memory_cell, nn.LSTM):
            packed_outputs, (hidden_layers, cell_layers) = self.memory_cell(packed_embedding)
        else:
            packed_outputs, hidden_layers = self.memory_cell(packed_embedding)
            
        # hidden_layers shape: (num_layers * num_directions, B, hidden_dim)

        if self.representation == "final":
            # Handle bidirectional stacking
            if self.bidirectional:
                # -2 is the last forward layer, -1 is the last backward layer
                forward_hidden = hidden_layers[-2]
                backward_hidden = hidden_layers[-1]
                final_representation = torch.cat([forward_hidden, backward_hidden], dim=1)
            else:
                final_representation = hidden_layers[-1]
                
        elif self.representation == "mean":
            # Unpack
            # CRITICAL FIX: total_length ensures output matches input_ids shape even if batch max len is shorter
            padded_outputs, _ = pad_packed_sequence(
                packed_outputs, 
                batch_first=True, 
                total_length=input_ids.size(1) 
            ) 
            
            # Masking
            mask = (input_ids != self.padding_idx).unsqueeze(-1).float() 
            masked_sum = (mask * padded_outputs).sum(dim=1)
            
            # Division (lengths is on device because it came from x)
            # Clamp lengths to avoid division by zero
            final_representation = masked_sum / lengths.unsqueeze(1).float().clamp(min=1.0)
            
        return self.FC(final_representation)

# --- Setup ---

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")

# Synthetic Data 
vocab_size = 1000
dataset_size = 10000
seq_len = 20
padding_idx = 0 
embed_dim = 256
num_classes = 2 
batch_size = 64

X = torch.randint(1, vocab_size, (dataset_size, seq_len))
all_paddings = torch.randint(0, seq_len//2, (dataset_size,))
for i, padding in enumerate(all_paddings):
    pad = padding.item()
    if pad > 0:
        X[i, (seq_len - pad):] = padding_idx 
Y = (X >= 450).any(dim=1).long()

X_train, X_val, X_test = torch.split(X, [8000,1000,1000])
Y_train, Y_val, Y_test = torch.split(Y, [8000,1000,1000])

train_dataset = TensorDataset(X_train, Y_train)
val_dataset = TensorDataset(X_val, Y_val)
test_dataset = TensorDataset(X_test, Y_test)

# Optimization: Use num_workers > 0
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=2) 

# Hyperparameters
hidden_dim = 128
num_layers = 1
bidirectional = True # Changed to True to test that logic
representation = "mean" # Changed to mean to test that logic
epochs = 10 
patience = 3

# model = RNNClassifier(
#     vocab_size, embed_dim, padding_idx, nn.LSTM, # Passing LSTM class
#     hidden_dim, num_layers, bidirectional, representation, num_classes
# ).to(device)
# model = RNNClassifier(
#     vocab_size, embed_dim, padding_idx, nn.GRU, # Passing LSTM class
#     hidden_dim, num_layers, bidirectional, representation, num_classes
# ).to(device)
model = RNNClassifier(
    vocab_size, embed_dim, padding_idx, nn.RNN, # Passing LSTM class
    hidden_dim, num_layers, bidirectional, representation, num_classes
).to(device)



optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5) 
criterion = nn.CrossEntropyLoss()

def get_total_accuracy(logits, label):
    preds = torch.argmax(logits, dim=1)
    return (preds == label).float().sum().item()

# Training Loop
writer = SummaryWriter(log_dir="runs/model_runs")
global_step = 0
best_state = None
min_loss = float("inf")
counter = 0

for epoch in range(epochs):
    model.train()
    avg_train_acc = 0
    
    for x, y in train_loader:
        optimizer.zero_grad()
        x, y = x.to(device), y.to(device)
        
        # Calculate lengths and clamp to avoid 0
        lengths = (x != padding_idx).sum(dim=1).clamp(min=1)
        
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        
        # Optimization: clip_grad_norm_ returns the norm, no need to recalc
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        
        optimizer.step()
        
        avg_train_acc += get_total_accuracy(logits, y) 
        writer.add_scalar("train/grad_norm", grad_norm, global_step) 
        writer.add_scalar("train/loss", loss.item(), global_step) 
        global_step += 1

    avg_train_acc /= len(train_dataset)
    writer.add_scalar("train/avg_acc", avg_train_acc, epoch+1) 

    # Validation
    avg_val_acc = 0
    avg_val_loss = 0 
    model.eval()
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            lengths = (x != padding_idx).sum(dim=1).clamp(min=1)
            
            logits = model(x, lengths)
            loss = criterion(logits, y) 
            
            avg_val_acc += get_total_accuracy(logits, y)
            avg_val_loss += loss.item()

    avg_val_acc /= len(val_dataset)
    avg_val_loss /= len(val_loader)
    
    writer.add_scalar("val/avg_acc", avg_val_acc, epoch+1)
    writer.add_scalar("val/avg_loss", avg_val_loss, epoch+1)
    writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], epoch+1) 

    print(f"Epoch {epoch+1}: Train Acc {avg_train_acc:.4f} | Val Acc {avg_val_acc:.4f} | Val Loss {avg_val_loss:.4f}")
    
    scheduler.step(avg_val_loss)

    if avg_val_loss < min_loss:
        min_loss = avg_val_loss
        best_state = deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

if best_state is not None:
    model.load_state_dict(best_state)

# Testing
model.eval()
avg_test_acc = 0
avg_test_loss = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        lengths = (x != padding_idx).sum(dim=1).clamp(min=1)
        
        logits = model(x, lengths)
        loss = criterion(logits, y)
        
        avg_test_loss += loss.item()
        avg_test_acc += get_total_accuracy(logits, y)

avg_test_acc /= len(test_dataset)
avg_test_loss /= len(test_loader)

print(f"Final Test Loss: {avg_test_loss:.4f} | Accuracy: {avg_test_acc:.4f}")

%load_ext tensorboard
%tensorboard --logdir runs --port 6007


## Transformer

#### Rotary Attention 

In [2]:
import math 
import torch
from torch import nn 

# apply rotary attention 

class RotarySelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.query = nn.Linear(d_model, d_model) # query for the input token 
        self.key = nn.Linear(d_model, d_model) # key for the input token 
        self.value = nn.Linear(d_model, d_model) # value for all tokens 

    def apply_rotary(self, x_inputs):
        # used to rotate query and key matrix, query for position m, key for position n in the entire seq 
        # formula after dot protate with rotation simplies to cos(m-n) = cosmthetacosntheta + sinmthetasinntheta

        B, S, D = x_inputs.shape 
        # D needs to be even since the calculation uses
        # x, y  as even and odd position, x_new = x*cos - ysin, y_new = xsin + ycos
        if D % 2:
            raise ValueError(f"feature dimension needs to be even not odd {D}")
        x_inputs = x_inputs.view(B, S, D//2, 2) # split into even position x and odd position y 

        # define theta freq for closer dimension with large angle and further out with small angle for position 
        freqs = torch.arange(D//2, device=x_inputs.device, dtype=x_inputs.dtype) # (D//2,) [0, 1, ...D//2-1] # creates theta 
        thetas = 10000**(-2*freqs/D) ## thetas for rotation for each feature dimension (D//2,)
        positions = torch.arange(S, device=x_inputs.device, dtype=x_inputs.dtype) # (S,) [0,1,..S-1] # use this to calculate angles
        angles = thetas.view(1, D//2) * positions.view(S,1) # (S,D//2) the angle is calcuated based upon feature dimension * token position in seq
        cos = angles.cos().view(1,S,D//2,1) 
        sin = angles.sin().view(1,S,D//2,1) 

        # we can apply angle rotation for the inputs 
        rotated_inputs = torch.zeros_like(x_inputs) # (B, S, D//2, 2) 
        even_inputs = x_inputs[...,0:1] # even positions for feature dimension 
        odd_inputs = x_inputs[...,1:2] # odd positions for feature dimension 
        # calcuate new even and odd position for the rotated inputs 
        rotated_inputs[...,0:1] = even_inputs*cos - odd_inputs*sin # (B, S, D//2, 1) 
        rotated_inputs[...,1:2] = even_inputs*sin + odd_inputs*cos # (B, S, D//2, 1) 
        return rotated_inputs.view(B, S, D) 

    def forward(self, embeddings, mask=None): # mask for decoder, autogregressive 
        B, S, D = embeddings.shape
        query = self.query(embeddings) # (B,S,D), input token 
        key = self.key(embeddings) # (B,S,D), all token in seq
        value = self.value(embeddings) # (B,S,D), all token in seq for the value 

        # apply positional rotation 
        query = self.apply_rotary(query) # position for the input token m 
        key = self.apply_rotary(key) # position for the all token seq in n 

        # apply dot product to get cos(m-n)theta 
        weighted_atten = torch.matmul(query, key.transpose(1,2))/math.sqrt(D) # (B,S,S) given normed input has var 1, without sqrt(D), variance D, cause softmax saturation 

        # apply mask for decoder 
        if mask is not None:
            mask = mask.view(B,1,S) # (B,S) -> (B,1,S) # dim=2 is the key seq dimension 
            weighted_atten = weighted_atten.masked_fill(mask == 0, float("-inf"))
        weighted_atten = torch.softmax(weighted_atten, dim=2) # (B,S,S), dim=2 is along the key sequence dim 

        # calculate final output 
        self_attention = torch.matmul(weighted_atten, value) # (B,S,D) 

        return self_attention
        

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h, dropout):
        super().__init__()
        if d_model % h:
            raise ValueError(f"d_model {d_model} need to be divisible by number of heads {h}") 
        self.d_model = d_model
        self.h = h 
        self.layer_norm = nn.LayerNorm(d_model)
        self.h_d = d_model // h 
        self.attention_heads = nn.ModuleList(
            [RotarySelfAttention(self.h_d) for _ in range(h)]
        )
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(d_model, d_model) # after concat all heads

    def forward(self, embeddings, mask=None):
        normed_embeddings = self.layer_norm(embeddings) # layernorm before multi head attentions
        attention_heads = []
        for i, attention in enumerate(self.attention_heads):
            attention_heads.append(
                attention(normed_embeddings[...,i*self.h_d: (i+1)*self.h_d], mask)
            )

        attention_outputs = torch.cat(attention_heads, dim=2)
        projection = self.proj(attention_outputs) 
        return self.dropout(projection) + embeddings 

## More efficient way of parallelize multi head attention 

In [11]:
import math 
import torch
from torch import nn 

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, dropout, h):
        super().__init__()
        if d_model % h:
            raise ValueError(f"d_model {d_model} need to be divisible by number of heads {h}") 
        self.h = h 
        self.d_h = d_model//h 
        self.layer_norm = nn.LayerNorm(d_model)
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout) 

    def apply_rotary(self, x_inputs):
        # used to rotate query and key matrix, query for position m, key for position n in the entire seq 
        # formula after dot protate with rotation simplies to cos(m-n) = cosmthetacosntheta + sinmthetasinntheta

        B, H, S, D = x_inputs.shape 
        # D needs to be even since the calculation uses
        # x, y  as even and odd position, x_new = x*cos - ysin, y_new = xsin + ycos
        if D % 2:
            raise ValueError(f"feature dimension needs to be even not odd {D}")
        x_inputs = x_inputs.view(B, H, S, D//2, 2) # split into even position x and odd position y 

        # define theta freq for closer dimension with large angle and further out with small angle for position 
        freqs = torch.arange(D//2, device=x_inputs.device, dtype=x_inputs.dtype) # (D//2,) [0, 1, ...D//2-1] # creates theta 
        thetas = 10000**(-2*freqs/D) ## thetas for rotation for each feature dimension (D//2,)
        positions = torch.arange(S, device=x_inputs.device, dtype=x_inputs.dtype) # (S,) [0,1,..S-1] # use this to calculate angles
        angles = thetas.view(1, D//2) * positions.view(S,1) # (S,D//2) the angle is calcuated based upon feature dimension * token position in seq
        cos = angles.cos().view(1,1, S,D//2,1) 
        sin = angles.sin().view(1,1, S,D//2,1) 

        # we can apply angle rotation for the inputs 
        rotated_inputs = torch.zeros_like(x_inputs) # (B, H, S, D//2, 2) 
        even_inputs = x_inputs[...,0:1] # even positions for feature dimension 
        odd_inputs = x_inputs[...,1:2] # odd positions for feature dimension 
        # calcuate new even and odd position for the rotated inputs 
        rotated_inputs[...,0:1] = even_inputs*cos - odd_inputs*sin # (B, H, S, D//2, 1) 
        rotated_inputs[...,1:2] = even_inputs*sin + odd_inputs*cos # (B,H,  S, D//2, 1) 
        return rotated_inputs.view(B, H, S, D) 
 

    def forward(self, embeddings, causal_mask=None, padding_mask=None):
        B, S, D = embeddings.shape 
        normed_embeddings = self.layer_norm(embeddings) 
        query = self.query(normed_embeddings)
        key = self.key(normed_embeddings)
        value = self.value(normed_embeddings) 

        query = query.view(B, self.h, S, self.d_h) 
        key = key.view(B, self.h, S, self.d_h)
        value = value.view(B, self.h, S, self.d_h) 

        query = self.apply_rotary(query)
        key = self.apply_rotary(key)

        attentions = torch.matmul(query, key.transpose(2,3))/math.sqrt(self.d_h) # (B,H,S,S) 
        if causal_mask is not None:
            attentions = attentions.masked_fill(causal_mask == 0, float("-inf"))
        if padding_mask is not None:
            padding_mask = padding_mask.view(B,1,1,S)
            attentions = attentions.masked_fill(padding_mask == 0, float("-inf"))
        attentions = torch.softmax(attentions, dim=3) # (B, H, S, S) 
        weighted_attentions = torch.matmul(attentions, value).view(B, S, D)  # (B, S, D) 
        projection = self.proj(weighted_attentions)
        return self.dropout(projection) + embeddings 

In [12]:
class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.layers = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, d_model),
            nn.Dropout(dropout)
        )
        
    def forward(self, atten_outputs):
        outputs = self.layers(atten_outputs)
        return outputs + atten_outputs     

In [13]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, hidden_dim, h, dropout):
        super().__init__()
        self.attention_outputs = MultiHeadAttention(d_model, dropout, h) 
        self.ffn = FeedForward(d_model, hidden_dim, dropout) 

    def forward(self, x_inputs, causal_mask=None, padding_mask=None):
        attention_outputs = self.attention_outputs(x_inputs, causal_mask, padding_mask) 
        ffn_outputs = self.ffn(attention_outputs)
        return ffn_outputs 

## Encoder Only Model

In [14]:
class Bert(nn.Module):
    def __init__(self, vocab_size, d_model, padding_idx, h, n, hidden_dim, dropout):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, hidden_dim, h, dropout)
            for _ in range(n)
        ])
        self.layer_norm = nn.LayerNorm(d_model) 
        self.linear = nn.Linear(d_model, vocab_size)
        self.linear.weight = self.embeddings.weight

    def forward(self, input_ids, padding_mask=None):
        embeddings = self.embeddings(input_ids)
        transformer_outputs = embeddings
        for block in self.transformer_blocks:
            transformer_outputs = block(transformer_outputs, None, padding_mask)
        normed_outputs = self.layer_norm(transformer_outputs)
        return self.linear(normed_outputs) 

In [21]:
from torch.utils.data import Dataset
from transformers import BertTokenizer
model_name = "bert-base-uncased" 
tokenizer = BertTokenizer.from_pretrained(model_name)

class BERTDataset(Dataset):
    def __init__(self, sentences: list, tokenizer: BertTokenizer, max_len: int, mask_pct:float):
        self.pad_token_id = tokenizer.pad_token_id
        self.mask_token_id = tokenizer.mask_token_id
        self.max_len = max_len 
        self.tokenizer = tokenizer 
        self.mask_pct = mask_pct
        self.datasets = self._generate_dataset(sentences)

    def _tokenize(self, sentence:str) -> torch.Tensor:
        encoded_input = self.tokenizer(sentence, max_length=self.max_len, truncation=True, return_tensors='pt').input_ids[0]
        if len(encoded_input) < self.max_len:
            encoded_input = torch.cat([encoded_input, torch.tensor([self.pad_token_id]*(self.max_len - len(encoded_input)))])
        return encoded_input[:self.max_len]

    def _mlm_mask(self, token_ids: torch.Tensor) -> tuple:
        n = len(token_ids)
        labels = torch.full((n,),-100, dtype=torch.long)

        special_tokens_mask = (
            (token_ids == self.tokenizer.pad_token_id) | 
            (token_ids == self.tokenizer.cls_token_id) | 
            (token_ids == self.tokenizer.sep_token_id)
        )

        candidate_indices = torch.nonzero(~special_tokens_mask, as_tuple=False).view(-1)
        padding_mask = (token_ids != self.pad_token_id).long()
        num_to_mask = int(self.mask_pct*len(candidate_indices))
        if num_to_mask == 0:
            return (token_ids, labels, padding_mask)
        masked_indices = candidate_indices[torch.randperm(len(candidate_indices))[:num_to_mask]]
        masked_token_ids = token_ids.clone()
        for idx in masked_indices:
            masked_token_ids[idx] = self.mask_token_id
            labels[idx] = token_ids[idx]
        return (masked_token_ids, labels, padding_mask)

    def _generate_dataset(self, sentences:list) -> list:
        all_datasets = []
        for sentence in sentences:
            tokenized = self._tokenize(sentence)
            all_datasets.append(self._mlm_mask(tokenized))
        return all_datasets

    def __len__(self):
        return len(self.datasets)

    def __getitem__(self, idx):
        return self.datasets[idx] 

In [22]:
from datasets import load_dataset
from torch.utils.data import DataLoader
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

train_sentences = [item["text"] for item in dataset["train"] if len(item["text"].strip()) > 0 ][:2000]
val_sentences = [item["text"] for item in dataset["validation"] if len(item["text"].strip()) > 0 ][:500]
test_sentences = [item["text"] for item in dataset["test"] if len(item["text"].strip()) > 0 ][:500]
batch_size = 4
mask_pct = 0.15
max_len = 128
train_dataset = BERTDataset(train_sentences, tokenizer, max_len, mask_pct)
val_dataset = BERTDataset(val_sentences, tokenizer, max_len, mask_pct)
test_dataset = BERTDataset(test_sentences, tokenizer, max_len, mask_pct)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [23]:
vocab_size = tokenizer.vocab_size
d_model = 768  
padding_idx = tokenizer.pad_token_id
h = 12
n = 12
hidden_dim = 768 * 4 
dropout = 0.1
total_epochs = 3 

model = Bert(vocab_size,d_model,padding_idx,h,n,hidden_dim,dropout)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
total_steps = len(train_sentences)*total_epochs//batch_size 
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps) 
from torch.optim.lr_scheduler import LambdaLR
warmup_steps = 0.1*total_steps 

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return current_step/warmup_steps 
    progress = (current_step - warmup_steps)/(total_steps - warmup_steps) 
    return 0.5 * (1 + math.cos(math.pi* progress))

scheduler = LambdaLR(optimizer, lr_lambda)

In [24]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter(log_dir="runs/bert-tests")

step = 0 
model = model.to(device)
for epoch in range(total_epochs):
    model.train()
    for x, y, mask in train_loader:
        optimizer.zero_grad()
        x = x.to(device)
        y = y.to(device)
        mask = mask.to(device)

        logits = model(x, padding_mask=mask)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 

        optimizer.step()
        scheduler.step()

        writer.add_scalar("train/loss", loss.item(), step)
        writer.add_scalar("train/grad_norm", grad_norm, step)
        writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], step)
        print(f"train loss {loss.item()} with learning rate {optimizer.param_groups[0]['lr']} for step {step}")
        step += 1

    model.eval()
    avg_val_loss = 0 
    with torch.no_grad():
        for x, y, mask in val_loader:
            x,y,mask = x.to(device), y.to(device), mask.to(device)
            logits = model(x, mask) 
            avg_val_loss += criterion(logits.view(-1,vocab_size), y.view(-1)).item()

    avg_val_loss /= len(val_loader)
    writer.add_scalar("val/loss", avg_val_loss, epoch) 

    print(f"Epoch {epoch} val_loss {avg_val_loss}")

writer.close() 
model.eval()
avg_test_loss = 0 
with torch.no_grad():
    for x, y, mask in test_loader:
        x,y,mask = x.to(device), y.to(device), mask.to(device)
        logits = model(x,mask).view(-1, vocab_size)
        avg_test_loss += criterion(logits, y.view(-1)).item()
avg_test_loss /= len(test_loader)
print(f"Test loss: {avg_test_loss}")

%load_ext tensorboard
%tensorboard --logdir runs --port 6007


train loss 584.8267822265625 with learning rate 6.666666666666667e-07 for step 0
train loss 577.8131103515625 with learning rate 1.3333333333333334e-06 for step 1
train loss 590.627685546875 with learning rate 2.0000000000000003e-06 for step 2
train loss 568.6077880859375 with learning rate 2.666666666666667e-06 for step 3
train loss 570.7224731445312 with learning rate 3.3333333333333333e-06 for step 4
train loss 550.9420776367188 with learning rate 4.000000000000001e-06 for step 5
train loss 549.1192016601562 with learning rate 4.666666666666667e-06 for step 6
train loss 524.3511962890625 with learning rate 5.333333333333334e-06 for step 7
train loss 494.76611328125 with learning rate 6e-06 for step 8
train loss 483.59991455078125 with learning rate 6.666666666666667e-06 for step 9
train loss 440.1972961425781 with learning rate 7.333333333333334e-06 for step 10
train loss 408.4139099121094 with learning rate 8.000000000000001e-06 for step 11
train loss 359.3639221191406 with learnin

Reusing TensorBoard on port 6007 (pid 4457), started 0:16:15 ago. (Use '!kill 4457' to kill it.)

## Decoder Only Model 

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, padding_idx, h, n, hidden_dim, dropout):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, hidden_dim, h, dropout)
            for _ in range(n)
        ])
        self.layer_norm = nn.LayerNorm(d_model) 
        self.linear = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, causal_mask=None, padding_mask=None):
        B, S = input_ids.shape 
        causal_mask = torch.tril(torch.ones((S,S), device=input_ids.device)).view(1, 1, S, S) 
        embeddings = self.embeddings(input_ids) # (B,S,D) 
        transformer_outputs = embeddings 
        for block in self.transformer_blocks:
            transformer_outputs = block(transformer_outputs, causal_mask, padding_mask)
        normed_outputs = self.layer_norm(transformer_outputs)
        return self.linear(normed_outputs) 

## T5 encoder + Decoder Model

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, vocab_size, d_model, padding_idx, h, n, hidden_dim, dropout):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, hidden_dim, h, dropout)
            for _ in range(n)
        ])

    def forward(self, input_ids, padding_mask=None):
        embeddings = self.embeddings(input_ids)
        transformer_outputs = embeddings
        for block in self.transformer_blocks:
            transformer_outputs = block(transformer_outputs, None, padding_mask)
        return transformer_outputs 

class CrossAttention(nn.Module):
    def __init__(self, d_model, h, dropout):
        super().__init__()
        self.h = h 
        self.decoder_norm = nn.LayerNorm(d_model)
        self.encoder_norm = nn.LayerNorm(d_model)
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.feedforward = nn.Linear(d_model, d_model) 
        self.dropout = nn.Dropout(dropout)

    def forward(self, decoder_input, encoder_output, padding_mask=None):
        B, S_dec, D = decoder_input.shape
        B,S_enc,D = encoder_output.shape
        normed_decoder_input = self.decoder_norm(decoder_input)
        normed_encoder_output = self.encoder_norm(encoder_output) 
        query = self.query(normed_decoder_input).view(B, self.h, S_dec, D//self.h) # (B, h, s_dec, h_d)
        key = self.key(normed_encoder_output).view(B, self.h, S_enc, D//self.h) # (B, h, s_enc, h_d)
        value = self.value(normed_encoder_output).view(B, self.h, S_enc, D//self.h) # (B, h, s_enc, h_d) 
        scaled_cross_attention = torch.matmul(query, key.transpose(2,3))/math.sqrt(D//self.h) # (B, h, s_dec, s_enc) 
        if padding_mask is not None:
            padding_mask = padding_mask.view(B, 1, 1, S_enc) 
            scaled_cross_attention = scaled_cross_attention.masked_fill(padding_mask==0, float("-inf"))
        scaled_cross_attention = torch.softmax(scaled_cross_attention, dim=3) 
        weighted_cross_attention = torch.matmul(scaled_cross_attention, value).view(B, S_dec, D) 
        project = self.feedforward(weighted_cross_attention)
        return self.dropout(project) + decoder_input 
        
    

class DecoderBlock(nn.Module):
    def __init__(self,  d_model,  h, hidden_dim, dropout):
        super().__init__()
        self.causal_self_attention = MultiHeadAttention(d_model, dropout, h) 
        self.cross_attention = CrossAttention(d_model, h, dropout)
        self.FFC = FeedForward(d_model, hidden_dim, dropout)

    def forward(self, x_inputs, encoder_output, padding_mask=None):
        B, S, D = x_inputs.shape 
        causal_mask = torch.tril(torch.ones((S,S), device=x_inputs.device)).view(1, 1, S, S) 
        causal_self_attention = self.causal_self_attention(x_inputs, causal_mask=causal_mask)
        cross_attention = self.cross_attention(causal_self_attention, encoder_output, padding_mask) 
        return self.FFC(cross_attention)


class T5(nn.Module):
    def __init__(self, vocab_size, d_model, padding_idx, n, h, hidden_dim, dropout):
        super().__init__() 
        self.decoder_embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.encoder = EncoderBlock(vocab_size, d_model, padding_idx, h, n, hidden_dim, dropout)
        self.decoder_blocks = nn.ModuleList([DecoderBlock(d_model, h, hidden_dim, dropout) for _ in range(n)])
        self.layer_norm = nn.LayerNorm(d_model) 
        self.linear = nn.Linear(d_model, vocab_size) 

    def forward(self, encoder_input_ids, decoder_input_ids, padding_mask=None):
        encoder_outputs = self.encoder(encoder_input_ids, padding_mask) 
        decoder_outputs = self.decoder_embedding(decoder_input_ids)
        for block in self.decoder_blocks:
            decoder_outputs = block(decoder_outputs, encoder_outputs, padding_mask) 
        normed_outputs = self.layer_norm(decoder_outputs) 
        return self.linear(normed_outputs) 
        